# Infer a liquid-neon Mie potential from LGPMD RDF data

This example uses the liquid-neon radial distribution function (RDF) data distributed with the [LGPMD `tutorial_v2.0`](https://github.com/hoepfnergroup/LGPMD/tree/main/tutorial_v2.0) directory. The files under `upstream/` were copied verbatim from commit `e2787cf0d830758f65f133fd1d2f7258a2ad3dee`.

The dataset accompanies Brennon L. Shanks, Harry Sullivan, Benjamin Shazed, and Michael P. Hoepfner, *Accelerated Bayesian Inference for Molecular Simulations using Local Gaussian Process Surrogate Models*, J. Chem. Theory Comput. (2024), [DOI: 10.1021/acs.jctc.3c01358](https://doi.org/10.1021/acs.jctc.3c01358).

We infer the three parameters of a lambda-6 Mie potential: `epsilon`, `lambda`, and `sigma`. No MD engine is needed because the simulation training data are already available.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from pickle import load

import matplotlib.pyplot as plt
import numpy as np
import torch

from bff.bayes.effective_observations import estimate_curve_n_eff
from bff.bayes.learning import LearningProblem, fit_surrogates
from bff.bayes.likelihoods import gaussian_log_likelihood_by_qoi
from bff.bayes.priors import Prior, Priors
from bff.domain.specs import ChargeConstraint, Specs
from bff.io.logs import Logger
from bff.plotting import plot_corner, plot_qoi_marginals
from bff.qoi.data import QoIDataset

ROOT = Path.cwd()
if not (ROOT / "upstream").exists():
    ROOT = Path("examples/neon-mie-lgpmd").resolve()
UPSTREAM = ROOT / "upstream"
GENERATED = ROOT / "generated"
MODEL_DIR = GENERATED / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

_ = torch.manual_seed(7)

## Load and align the upstream RDF data

LGPMD provides 480 simulated training RDFs, 160 held-out testing RDFs, and an experimental neon RDF at approximately 42 K. Following the upstream tutorial, we interpolate the simulation and experimental RDFs onto a shared 73-bin grid.

In [ ]:
with (UPSTREAM / "training_data/samples.p").open("rb") as handle:
    raw_inputs = load(handle)["xs"]
with (UPSTREAM / "training_data/training_rdf.p").open("rb") as handle:
    training_rdf = load(handle)
experimental_table = np.loadtxt(
    UPSTREAM / "exp_data/ne_42K_rdf_new.csv",
    delimiter=",",
    skiprows=1,
)
rdf_grid = np.linspace(0.0118331810091873, 15.512161254882812, 73)
simulated_rdfs = np.asarray([
    np.interp(rdf_grid, training_rdf["r"], values)
    for values in training_rdf["model_rdf"]
])
experimental_rdf = np.interp(
    rdf_grid, experimental_table[:, 0], experimental_table[:, 1]
)
experimental_rdf -= experimental_rdf.min()

print(f"training inputs: {raw_inputs.shape}")
print(f"training RDFs:   {simulated_rdfs.shape}")
print(f"RDF bins:        {rdf_grid.size}")

## Convert the data into BFF format

The three Mie parameters are stored and learned directly in their physical units. `Specs` provides the stable parameter order and physical bounds; an empty `charge_constraints` list is appropriate because this example does not fit partial charges. We retain the simulation rows inside this physical inference domain. This also removes the broad tutorial rows with $\lambda \leq 6$, for which a lambda-6 Mie PMF is undefined.

The source dataset does not include an experimental RDF uncertainty. We therefore leave `nuisance` unset so BFF learns the RDF discrepancy jointly with the Mie parameters.

In [ ]:
specs = Specs({
    "bounds": {
        "epsilon [kcal mol^-1]": [0, 0.136],
        "lambda": [6.1, 18],
        "sigma [angstrom]": [0.88, 3.32],
    },
    "charge_constraints": [],
})
parameter_names = specs.parameter_names(explicit_only=True)

# LGPMD stores columns as (lambda, sigma, epsilon); BFF uses sorted names.
source_column = {
    "lambda": 0,
    "sigma [angstrom]": 1,
    "epsilon [kcal mol^-1]": 2,
}
inputs = np.column_stack([
    raw_inputs[:, source_column[name]] for name in parameter_names
])
bounds = specs.bounds.array
in_domain = np.all((inputs >= bounds[:, 0]) & (inputs <= bounds[:, 1]), axis=1)
inputs = inputs[in_domain]
simulated_rdfs = simulated_rdfs[in_domain]
print(f"retained simulation rows: {in_domain.sum()}/{in_domain.size}")

dataset = QoIDataset(
    name="rdf",
    inputs=inputs,
    outputs=simulated_rdfs,
    outputs_ref=experimental_rdf,
    labels=("neon RDF at 42.2 K",),
    values_per_label=rdf_grid.size,
    settings={
        "r": rdf_grid.tolist(),
        "temperature_K": 42.2,
        "n_bins": rdf_grid.size,
        "r_range": [rdf_grid.min(), rdf_grid.max()],
    },
    metadata={
        "source": "https://github.com/hoepfnergroup/LGPMD/tree/main/tutorial_v2.0",
        "source_commit": "e2787cf0d830758f65f133fd1d2f7258a2ad3dee",
        "article_doi": "10.1021/acs.jctc.3c01358",
    },
)
dataset.write(GENERATED / "qoi-rdf.pt")
dataset

## Fit and validate the local GP surrogate

BFF fits a local Gaussian-process surrogate directly from the converted `QoIDataset`. As in the paper, the surrogate mean is the RDF approximation $g(r) = \exp[-V_{\mathrm{Mie}}(r) / k_B T]$ derived from the potential of mean force. This is parameter-dependent: BFF evaluates it for each Mie parameter set before learning the local correction. The example-specific callable below demonstrates how to supply such a mean without adding neon-specific physics to BFF itself. We reserve simulation rows for BFF's internal train/test validation and then perform an additional check against LGPMD's separate held-out testing files.

The hyperparameters are optimized in log space. The priors below use the physically informed centers and scales from the upstream LGPMD tutorial, reordered to match BFF's sorted parameter names.

In [ ]:
K_BOLTZMANN_KCAL_MOL_K = 0.00198720425864083


@dataclass(frozen=True)
class MieRDFPMFMean:
    r: np.ndarray
    temperature: float
    lambda_index: int
    sigma_index: int
    epsilon_index: int

    def __call__(self, parameters: torch.Tensor) -> torch.Tensor:
        parameters = torch.as_tensor(parameters)
        if parameters.dim() == 1:
            parameters = parameters.unsqueeze(0)
        r = torch.as_tensor(
            self.r, device=parameters.device, dtype=parameters.dtype
        )
        mie_lambda = parameters[:, self.lambda_index, None]
        sigma = parameters[:, self.sigma_index, None]
        epsilon = parameters[:, self.epsilon_index, None]
        prefactor = (
            mie_lambda / (mie_lambda - 6)
            * (mie_lambda / 6) ** (6 / (mie_lambda - 6))
        )
        potential = prefactor * epsilon * (
            (sigma / r) ** mie_lambda - (sigma / r) ** 6
        )
        return torch.exp(
            -potential / (K_BOLTZMANN_KCAL_MOL_K * self.temperature)
        )


pmf_mean = MieRDFPMFMean(
    r=rdf_grid,
    temperature=dataset.settings["temperature_K"],
    epsilon_index=parameter_names.index("epsilon [kcal mol^-1]"),
    lambda_index=parameter_names.index("lambda"),
    sigma_index=parameter_names.index("sigma [angstrom]"),
)
rdf_hyperpriors = Priors([
    Prior("normal", -3.7, 2.0, name="length_epsilon"),
    Prior("normal", 0.3, 2.0, name="length_lambda"),
    Prior("normal", -3.0, 2.0, name="length_sigma"),
    Prior("normal", -2.0, 2.0, name="width"),
    Prior("normal", -6.0, 3.0, name="noise"),
])

fit_logger = Logger("fit", fn_log=GENERATED / "fit.log", mode="w")
models = fit_surrogates(
    [dataset],
    y_means={"rdf": pmf_mean},
    hyperpriors={"rdf": rdf_hyperpriors},
    model_paths={"rdf": MODEL_DIR / "rdf.lgp"},
    reuse_models=False,
    n_hyper_max=200,
    committee_size=1,
    test_fraction=0.2,
    device="cuda",
    logger=fit_logger,
    max_iter=10000,
)
committee = models["rdf"]
committee.n_eff = estimate_curve_n_eff(experimental_rdf, tolerance=0.1)
print(f"RDF effective observations: {committee.n_eff:.3f}")
print(f"BFF split-validation SMAPE: {committee.error:.3f}%")

In [ ]:
with (UPSTREAM / "testing_data/test_samples.p").open("rb") as handle:
    test_inputs_raw = load(handle)["xs"]
with (UPSTREAM / "testing_data/test_data.p").open("rb") as handle:
    test_rdfs = load(handle)["totalResults"]

test_inputs = np.column_stack([
    test_inputs_raw[:, source_column[name]] for name in parameter_names
])
validation_error = committee.validate(
    torch.as_tensor(test_inputs, dtype=torch.float32),
    torch.as_tensor(np.asarray(test_rdfs), dtype=torch.float32),
)
print(f"held-out surrogate SMAPE: {validation_error:.3f}%")

## Learn Mie parameters

`LearningProblem.learn()` runs BFF's standard parameter-learning workflow. The notebook does not reproduce the custom inference setup from the LGPMD publication; it uses the same public BFF interface as any other externally supplied `QoIDataset`.

In [ ]:
problem = LearningProblem.from_models(models, constraint=ChargeConstraint(specs))
learn_logger = Logger("learn", fn_log=GENERATED / "learn.log", mode="w")
results = problem.learn(
    priors_disttype="normal",
    total_steps=10000,
    warmup=5000,
    progress_stride=1000,
    fn_checkpoint=GENERATED / "mcmc-checkpoint.pt",
    fn_posterior=GENERATED / "posterior.pt",
    fn_priors=GENERATED / "priors.pt",
    restart=False,
    device="cuda",
    logger=learn_logger,
    rhat_tol=1.01,
    ess_min=100,
)

## Report the posterior

The sampled Mie parameters already use physical force-field units. BFF samples the RDF discrepancy as `log_sigma_rdf` internally and automatically reports it as a positive `sigma_rdf` value.

In [ ]:
results.prepare_samples(discard=160, thin=4)
results.write_summary(GENERATED / "posterior-summary.yaml")

model_device = committee.lgps[0].X_train.device
theta = results.prepared_samples.copy()
theta[:, problem.n_params:] = np.log(theta[:, problem.n_params:])
contributions = {
    name: values.detach().cpu().numpy()
    for name, values in gaussian_log_likelihood_by_qoi(
        torch.as_tensor(theta, dtype=torch.float32, device=model_device),
        problem.to_torch(str(model_device)),
    ).items()
}

plot_corner(results, fn_out=GENERATED / "corner.png")
plot_qoi_marginals(
    results,
    specs,
    contributions,
    fn_out=GENERATED / "qoi-marginals.pdf",
)
results.summary()

In [ ]:
posterior_mean = results.prepared_samples.mean(axis=0)
inferred_rdf = committee.predict(
    torch.as_tensor(posterior_mean[None, :len(parameter_names)], dtype=torch.float32)
)[0].detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rdf_grid, experimental_rdf, label="experiment", lw=2.5)
ax.plot(rdf_grid, inferred_rdf, label="posterior-mean surrogate", lw=2.0)
ax.set(xlabel="r [angstrom]", ylabel="g(r)", xlim=(0, rdf_grid.max()))
ax.legend()
fig.tight_layout()
fig.savefig(GENERATED / "rdf-fit.png", dpi=180)
plt.show()